# Chapter 16 --- Complete Workflow: Classification on Adult Income

**Monograph:** *How to Build Policy-Governed AI Agents*

**Prerequisites:**
- All chapters 1--15 completed.
- `governed_agent/` library fully built.

**What you will do:**
- Wire ALL governed-agent components into a single end-to-end pipeline.
- Execute 4 phases (data, training, diagnostics, fairness) on the Adult Income dataset.
- Extract knowledge triples and ENM entries at every step.
- Verify accumulated claims geometrically.
- Generate a governance report with a release gate decision.

**Key idea:** A capstone chapter is not about new infrastructure --- it is about proving that every component from Parts I--IV composes into a working governed system.

## Preamble

In [ ]:
import os
import warnings
import shutil
warnings.filterwarnings("ignore")

import numpy as np
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"NumPy : {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Device : {DEVICE}")
print("Ready.")

## Import All Governed-Agent Components

This is the capstone: we import components from **every chapter** in Parts I--IV.

| Component | Chapter | Module |
|-----------|---------|--------|
| `SessionState` | Ch 3 | `governed_agent.session` |
| `execute_tool` | Ch 3 | `governed_agent.tools` |
| `extract_from_tool` | Ch 4 | `governed_agent.extractors` |
| `AgentMemory` | Ch 5 | `governed_agent.memory` |
| `ClaimVerifier` | Ch 8 | `governed_agent.verifier` |
| `PhaseRunner` | Ch 16 | `governed_agent.agents.data_agent` |

In [ ]:
# Chapter 3: Session and Tools
from knowlytix.harness.governance.session import SessionState

# Chapter 4: Three-tier extraction

# Chapter 5: Agent memory (GMS + ENM)
from knowlytix.harness.governance.memory import AgentMemory

# Chapter 8: Claim verification
from knowlytix.harness.governance.verifier import (
    ClaimVerifier, ClaimType, Claim,
)

# Chapter 16: Phase runner and predefined phases
from knowlytix.harness.governance.agents.data_agent import (
    PhaseRunner, PhaseResult,
    adult_data_phase, adult_training_phase,
    adult_diagnostics_phase, generate_governance_report,
)

print("All governed-agent components imported successfully.")
print(f"  SessionState fields: {list(SessionState.__dataclass_fields__.keys())}")
print(f"  ClaimTypes: {[ct.value for ct in ClaimType]}")
print(f"  PhaseRunner methods: {[m for m in dir(PhaseRunner) if not m.startswith('_')]}")

## Initialize the Governed Pipeline

Three components must be created before any phase can run:

1. **SessionState** --- mutable state shared across tool calls (Ch 3).
2. **AgentMemory** --- GMS-backed knowledge store with ENM (Ch 5).
3. **ClaimVerifier** --- typed verification against the knowledge store (Ch 8).

We ingest the data science practices knowledge document to seed the GMS store with best-practice triples.

In [3]:
import yaml

# Clean any previous store for reproducibility
STORE_PATH = "governance_monograph/lab_stores/ch16_adult_store"
if os.path.exists(STORE_PATH):
    shutil.rmtree(STORE_PATH)

KNOWLEDGE_PATH = "governed_agent/knowledge/data_science_practices.md"

# 1. Session state (Ch 3)
session = SessionState()
print(f"SessionState created: {len(session.datasets)} datasets, "
      f"{len(session.models)} models")

# 2. Agent memory with knowledge ingestion (Ch 5)
memory = AgentMemory(
    store_path=STORE_PATH,
    knowledge_path=KNOWLEDGE_PATH,
    device=str(DEVICE),
)
stats = memory.memory_stats()
print("\nAgentMemory initialized:")
print(f"  Entities:  {stats['entities']}")
print(f"  Relations: {stats['relations']}")
print(f"  Triples:   {stats['triples']}")
print(f"  ENM:       {stats['enm_entries']}")

# 3. Claim verifier (Ch 8)
verifier = ClaimVerifier(memory.store, device=str(DEVICE))
print(f"\nClaimVerifier: {len(verifier._dispatch)} claim type handlers")

# 4. Load policy rules (Ch 12)
with open("governed_agent/policies/responsible_ml.yaml") as f:
    policy_doc = yaml.safe_load(f)
policy_rules = policy_doc.get("rules", [])
print(f"Policy rules loaded: {len(policy_rules)} rules")

SessionState created: 0 datasets, 0 models

Ingesting: governed_agent/knowledge/data_science_practices.md
  Converted to markdown: 10917 chars
  Parsed: 24 ENM, 370 triples, 8 phase encoders

  First document — building GMS from scratch...
  GMS entities:  244
  GMS relations: 38
  GMS triples:   370



Training GMS: 244 entities, 38 relations, 370 triples
  Config: d_v=128, m=64, epochs=200, batch=256, neg=32
  Device: cuda


  Epoch    1/200  Loss: 7.8187  (L_pos=1.5684, L_neg=0.0049)


  Epoch   20/200  Loss: 7.6595  (L_pos=1.5247, L_neg=0.0042)


  Epoch   40/200  Loss: 7.5226  (L_pos=1.5035, L_neg=0.0041)


...

  Epoch  120/200  Loss: 6.9654  (L_pos=1.3989, L_neg=0.0041)


  Epoch  140/200  Loss: 6.7945  (L_pos=1.3470, L_neg=0.0024)


  Epoch  160/200  Loss: 6.6993  (L_pos=1.3510, L_neg=0.0028)


  Epoch  180/200  Loss: 6.5164  (L_pos=1.2884, L_neg=0.0029)


  Epoch  200/200  Loss: 6.4055  (L_pos=1.2780, L_neg=0.0022)

  Populating ENM register...
  ENM populated: 24 entries
  Store saved to governance_monograph/lab_stores/ch16_adult_store

  Ingestion complete: 244 entities, 370 triples, 24 ENM entries (22.1s)

AgentMemory initialized:
  Entities:  244
  Relations: 38
  Triples:   370
  ENM:       24

ClaimVerifier: 5 claim type handlers
Policy rules loaded: 10 rules


---
## Lab 16.1 --- Run the Full Governed Pipeline

We create a `PhaseRunner` and execute all four phases sequentially.  After each phase we print:
- Tools called and any policy blocks
- Triples extracted and ENM entries stored
- Elapsed time

After all phases we print cumulative statistics and run the ClaimVerifier on all accumulated claims.

In [4]:
# Create the PhaseRunner with policy rules
runner = PhaseRunner(session, memory, policy_rules=policy_rules)
print("PhaseRunner created with", len(policy_rules), "policy rules.\n")

# Collect results and claims across all phases
all_phase_results: list[PhaseResult] = []
all_claims: list[Claim] = []

PhaseRunner created with 10 policy rules.



### Phase 1: Data --- Load, Summarize, Correlate, Preprocess

Load the Adult Income dataset from OpenML, generate summary statistics, compute feature correlations and fit a preprocessing pipeline (median imputation, standard scaling, ordinal encoding).

> **Note on `[BackgroundTrain]` messages.**  After each extraction, `AgentMemory`
> triggers background GMS training to incorporate new triples into the knowledge
> graph.  These log messages confirm that the store is learning incrementally ---
> they are not errors.  The phase summary printed afterward is the pedagogically
> relevant output.

In [5]:
data_result = runner.run_phase("DataPhase", adult_data_phase())

print(f"=== {data_result.phase_name} ===")
print(f"  Tools called:  {data_result.tools_called}")
print(f"  Triples:       {data_result.triples_extracted}")
print(f"  ENM entries:   {data_result.enm_entries}")
print(f"  Policy blocks: {data_result.policy_blocks}")
print(f"  Claims:        {len(data_result.claims)}")
print(f"  Elapsed:       {data_result.elapsed_s:.1f}s")

# Show per-tool trace
print("\nPer-tool trace:")
for entry in data_result.trace:
    status = "BLOCKED" if entry.get("blocked") else "OK"
    print(f"  {entry['tool']:20s} | {status:7s} | "
          f"triples={entry.get('triples', 0):3d} | "
          f"enm={entry.get('enm', 0):3d} | "
          f"tier={entry.get('tier', '-')}")

all_phase_results.append(data_result)
all_claims.extend(data_result.claims)

  [BackgroundTrain] Starting: 16 entities, 6 relations, 15 triples


  [BackgroundTrain] Done: Trained: 16 entities, 6 relations, 50 steps in 4.5s
  Store saved to governance_monograph/lab_stores/ch16_adult_store
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 2 entities, 3 relations, 3 triples


  [BackgroundTrain] Done: Trained: 2 entities, 3 relations, 50 steps in 3.3s
  Store saved to governance_monograph/lab_stores/ch16_adult_store
  [BackgroundTrain] Store saved
=== DataPhase ===
  Tools called:  ['load_dataset', 'data_summary', 'eda_correlation', 'preprocess_data']
  Triples:       18
  ENM entries:   7
  Policy blocks: 0
  Claims:        10
  Elapsed:       8.0s

Per-tool trace:
  load_dataset         | OK      | triples= 15 | enm=  4 | tier=1
  data_summary         | OK      | triples=  0 | enm=  0 | tier=3
  eda_correlation      | OK      | triples=  0 | enm=  0 | tier=3
  preprocess_data      | OK      | triples=  3 | enm=  3 | tier=2


### Phase 2: Training --- Train 5 Models and Produce Leaderboard

Train LogisticRegression, RandomForest, GradientBoosting, DecisionTree and SVC on the preprocessed Adult Income data.  All models use `random_state=42` for reproducibility.  The leaderboard ranks them by test accuracy.

Note that the policy engine requires preprocessing before training (RML-001) and at least 2 models before a leaderboard call (RML-004).

In [6]:
training_result = runner.run_phase("TrainingPhase", adult_training_phase())

print(f"=== {training_result.phase_name} ===")
print(f"  Tools called:  {training_result.tools_called}")
print(f"  Triples:       {training_result.triples_extracted}")
print(f"  ENM entries:   {training_result.enm_entries}")
print(f"  Policy blocks: {training_result.policy_blocks}")
print(f"  Claims:        {len(training_result.claims)}")
print(f"  Elapsed:       {training_result.elapsed_s:.1f}s")

# Show per-tool trace
print("\nPer-tool trace:")
for entry in training_result.trace:
    status = "BLOCKED" if entry.get("blocked") else "OK"
    print(f"  {entry['tool']:20s} | {status:7s} | "
          f"triples={entry.get('triples', 0):3d} | "
          f"enm={entry.get('enm', 0):3d} | "
          f"tier={entry.get('tier', '-')}")

all_phase_results.append(training_result)
all_claims.extend(training_result.claims)

# Show leaderboard from session
print("\nModel results (sorted by test_accuracy):")
ranked = sorted(session.results.items(),
                key=lambda kv: kv[1].get("test_accuracy", 0),
                reverse=True)
for mname, metrics in ranked:
    if mname in session.models:
        acc = metrics.get("test_accuracy", "N/A")
        f1 = metrics.get("test_f1_macro", "N/A")
        print(f"  {mname:5s}: accuracy={acc}, f1={f1}")

  [BackgroundTrain] Starting: 7 entities, 8 relations, 9 triples


  [BackgroundTrain] Done: Trained: 7 entities, 8 relations, 50 steps in 3.4s
  Store saved to governance_monograph/lab_stores/ch16_adult_store
  [BackgroundTrain] Store saved


  [BackgroundTrain] Starting: 8 entities, 0 relations, 9 triples


  [BackgroundTrain] Done: Trained: 8 entities, 0 relations, 50 steps in 3.9s
  Store saved to governance_monograph/lab_stores/ch16_adult_store
  [BackgroundTrain] Store saved


  [BackgroundTrain] Starting: 9 entities, 0 relations, 9 triples


  [BackgroundTrain] Done: Trained: 9 entities, 0 relations, 50 steps in 3.5s
  Store saved to governance_monograph/lab_stores/ch16_adult_store
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 9 entities, 0 relations, 9 triples


  [BackgroundTrain] Done: Trained: 9 entities, 0 relations, 50 steps in 3.3s
  Store saved to governance_monograph/lab_stores/ch16_adult_store
  [BackgroundTrain] Store saved


  [BackgroundTrain] Starting: 8 entities, 0 relations, 9 triples


  [BackgroundTrain] Done: Trained: 8 entities, 0 relations, 50 steps in 3.3s
  Store saved to governance_monograph/lab_stores/ch16_adult_store
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 6 entities, 2 relations, 46 triples


  [BackgroundTrain] Done: Trained: 6 entities, 2 relations, 50 steps in 3.8s
  Store saved to governance_monograph/lab_stores/ch16_adult_store
  [BackgroundTrain] Store saved
=== TrainingPhase ===
  Tools called:  ['train_model', 'train_model', 'train_model', 'train_model', 'train_model', 'leaderboard']
  Triples:       91
  ENM entries:   75
  Policy blocks: 0
  Claims:        91
  Elapsed:       153.3s

Per-tool trace:
  train_model          | OK      | triples=  9 | enm=  7 | tier=1
  train_model          | OK      | triples=  9 | enm=  7 | tier=1
  train_model          | OK      | triples=  9 | enm=  7 | tier=1
  train_model          | OK      | triples=  9 | enm=  7 | tier=1
  train_model          | OK      | triples=  9 | enm=  7 | tier=1
  leaderboard          | OK      | triples= 46 | enm= 40 | tier=1

Model results (sorted by test_accuracy):
  GB   : accuracy=0.844, f1=0.7434
  DT   : accuracy=0.8357, f1=0.7268
  LR   : accuracy=0.8185, f1=0.6958
  RF   : accuracy=0.8086, f1=0

### Identify Top 2 Models from the Leaderboard

Extract the top 2 models by test accuracy from the leaderboard to use for diagnostics and fairness phases.

In [7]:
# Identify top 2 models by test accuracy
model_accs = {
    mname: session.results[mname].get("test_accuracy", 0)
    for mname in session.models
}
top2 = sorted(model_accs, key=model_accs.get, reverse=True)[:2]
top1 = top2[0]

print(f"Top 2 models: {top2}")
print(f"  {top2[0]}: accuracy = {model_accs[top2[0]]}")
print(f"  {top2[1]}: accuracy = {model_accs[top2[1]]}")
print(f"\nTop model for fairness analysis: {top1}")

Top 2 models: ['GB', 'DT']
  GB: accuracy = 0.844
  DT: accuracy = 0.8357

Top model for fairness analysis: GB


### Phase 3: Diagnostics --- Robustness and Feature Importance

For each of the top 2 models: add Gaussian noise at 4 levels and re-score (robustness), then compute permutation feature importance on the test set.

In [8]:
diag_result = runner.run_phase(
    "DiagnosticsPhase",
    adult_diagnostics_phase(top_models=top2),
)

print(f"=== {diag_result.phase_name} ===")
print(f"  Tools called:  {diag_result.tools_called}")
print(f"  Triples:       {diag_result.triples_extracted}")
print(f"  ENM entries:   {diag_result.enm_entries}")
print(f"  Policy blocks: {diag_result.policy_blocks}")
print(f"  Claims:        {len(diag_result.claims)}")
print(f"  Elapsed:       {diag_result.elapsed_s:.1f}s")

# Show per-tool trace
print("\nPer-tool trace:")
for entry in diag_result.trace:
    status = "BLOCKED" if entry.get("blocked") else "OK"
    print(f"  {entry['tool']:20s} | {status:7s} | "
          f"triples={entry.get('triples', 0):3d} | "
          f"enm={entry.get('enm', 0):3d}")

all_phase_results.append(diag_result)
all_claims.extend(diag_result.claims)

  [BackgroundTrain] Starting: 4 entities, 7 relations, 7 triples


  [BackgroundTrain] Done: Trained: 4 entities, 7 relations, 50 steps in 3.4s
  Store saved to governance_monograph/lab_stores/ch16_adult_store
  [BackgroundTrain] Store saved


  [BackgroundTrain] Starting: 8 entities, 7 relations, 8 triples


  [BackgroundTrain] Done: Trained: 8 entities, 7 relations, 50 steps in 3.5s
  Store saved to governance_monograph/lab_stores/ch16_adult_store
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 2 entities, 0 relations, 7 triples


  [BackgroundTrain] Done: Trained: 2 entities, 0 relations, 50 steps in 3.4s
  Store saved to governance_monograph/lab_stores/ch16_adult_store
  [BackgroundTrain] Store saved


  [BackgroundTrain] Starting: 6 entities, 0 relations, 8 triples


  [BackgroundTrain] Done: Trained: 6 entities, 0 relations, 50 steps in 4.0s
  Store saved to governance_monograph/lab_stores/ch16_adult_store
  [BackgroundTrain] Store saved
=== DiagnosticsPhase ===
  Tools called:  ['diagnose_robustness', 'explain_importance', 'diagnose_robustness', 'explain_importance']
  Triples:       30
  ENM entries:   20
  Policy blocks: 0
  Claims:        26
  Elapsed:       16.2s

Per-tool trace:
  diagnose_robustness  | OK      | triples=  7 | enm=  4
  explain_importance   | OK      | triples=  8 | enm=  6
  diagnose_robustness  | OK      | triples=  7 | enm=  4
  explain_importance   | OK      | triples=  8 | enm=  6


### Phase 4: Fairness --- AIR on Sex and Race

Compute Adverse Impact Ratio (AIR) and demographic parity difference for the top model on two protected attributes: sex and race.  The 4/5 rule requires AIR >= 0.80.

We run sex and race as separate sub-phases so that we can capture both results
before `session.results` is overwritten (the session stores only the last fairness
result per model).  The two sub-phases are merged into a single `FairnessPhase`
result for cumulative statistics and reporting.

In [ ]:
# Run sex fairness first, capture before race overwrites it
sex_tools = [("diagnose_fairness", {
    "dataset": "adult", "model_name": top1, "protected_attribute": "sex"})]
sex_result = runner.run_phase("Fairness_sex", sex_tools)
sex_fair = session.results.get(f"{top1}_fairness", {}).copy()

# Run race fairness
race_tools = [("diagnose_fairness", {
    "dataset": "adult", "model_name": top1, "protected_attribute": "race"})]
race_result = runner.run_phase("Fairness_race", race_tools)
race_fair = session.results.get(f"{top1}_fairness", {})

# Merge into a single FairnessPhase result for downstream reporting
from knowlytix.harness.governance.agents.data_agent import PhaseResult
fair_result = PhaseResult(
    phase_name="FairnessPhase",
    tools_called=sex_result.tools_called + race_result.tools_called,
    triples_extracted=sex_result.triples_extracted + race_result.triples_extracted,
    enm_entries=sex_result.enm_entries + race_result.enm_entries,
    policy_blocks=sex_result.policy_blocks + race_result.policy_blocks,
    trace=sex_result.trace + race_result.trace,
    claims=sex_result.claims + race_result.claims,
    elapsed_s=round(sex_result.elapsed_s + race_result.elapsed_s, 2),
)

print(f"=== {fair_result.phase_name} ===")
print(f"  Tools called:  {fair_result.tools_called}")
print(f"  Triples:       {fair_result.triples_extracted}")
print(f"  ENM entries:   {fair_result.enm_entries}")
print(f"  Policy blocks: {fair_result.policy_blocks}")
print(f"  Claims:        {len(fair_result.claims)}")
print(f"  Elapsed:       {fair_result.elapsed_s:.1f}s")

# Show both sex and race results
for label, fair in [("sex", sex_fair), ("race", race_fair)]:
    attr = fair.get("protected_attribute", label)
    air = fair.get("adverse_impact_ratio", "N/A")
    passes = fair.get("air_passes_4_5_rule", "N/A")
    dp = fair.get("demographic_parity_diff", "N/A")
    print(f"\n  {top1} -- {attr}:")
    print(f"    AIR = {air}, passes 4/5 rule = {passes}")
    print(f"    Demographic parity diff = {dp}")
    rates = fair.get("group_positive_rates", {})
    if rates:
        print(f"    Group rates: {rates}")

all_phase_results.append(fair_result)
all_claims.extend(fair_result.claims)

### Cumulative Statistics

Now that all four phases are complete, print the cumulative pipeline statistics and the final state of the GMS memory store.

In [10]:
cumulative = runner.cumulative_stats()

print("=== Cumulative Pipeline Statistics ===\n")
for key, val in cumulative.items():
    print(f"  {key:25s}: {val}")

print("\n=== Claims Accumulated ===")
print(f"  Total claims: {len(all_claims)}")
claim_types = {}
for c in all_claims:
    claim_types[c.type.value] = claim_types.get(c.type.value, 0) + 1
for ct, count in sorted(claim_types.items()):
    print(f"    {ct:12s}: {count}")

=== Cumulative Pipeline Statistics ===

  total_tools              : 16
  total_triples            : 156
  total_enm_entries        : 113
  total_policy_blocks      : 0
  memory_entities          : 342
  memory_relations         : 82
  memory_triples           : 526
  memory_enm               : 124

=== Claims Accumulated ===
  Total claims: 142
    fact        : 29
    numeric     : 113


### Verify All Claims Geometrically

Run the ClaimVerifier from Chapter 8 on every accumulated claim.  Each claim is routed to the appropriate GMS primitive: `score_triple` for facts, `lookup_enm` for numerics.

In [11]:
# Verify all claims
overall_verdict = verifier.verify_all(all_claims)

print("=== Verification Results ===\n")
print(ClaimVerifier.format_overall(overall_verdict))

# Show severity distribution
print("\nSeverity distribution:")
for sev, count in sorted(overall_verdict.severity_distribution.items()):
    print(f"  {sev:10s}: {count}")

# Show a sample of verified claims (first 5 passed, first 5 failed)
passed_verdicts = [v for v in overall_verdict.claim_verdicts if v.passed]
failed_verdicts = [v for v in overall_verdict.claim_verdicts if not v.passed]

print(f"\n--- Sample Passed Claims ({len(passed_verdicts)} total) ---")
for v in passed_verdicts[:5]:
    print(ClaimVerifier.format_verdict(v))

if failed_verdicts:
    print(f"\n--- Sample Failed Claims ({len(failed_verdicts)} total) ---")
    for v in failed_verdicts[:5]:
        print(ClaimVerifier.format_verdict(v))

=== Verification Results ===

=== Overall Verdict: PASS ===
  Passed: 142/142 (100.0%)
  Severity distribution: {'info': 142}

Severity distribution:
  info      : 142

--- Sample Passed Claims (142 total) ---
  [+] numeric      | PASS | score=1.000 | severity=info     | adult_n_samples = 48842.0
      evidence: lookup_enm(data, adult_n_samples) = 48842.000000, claimed = 48842.000000, rel_diff = 0.000000 <= 0.01
  [+] numeric      | PASS | score=1.000 | severity=info     | adult_n_features = 14.0
      evidence: lookup_enm(data, adult_n_features) = 14.000000, claimed = 14.000000, rel_diff = 0.000000 <= 0.01
  [+] numeric      | PASS | score=1.000 | severity=info     | adult_train_size = 39073.0
      evidence: lookup_enm(data, adult_train_size) = 39073.000000, claimed = 39073.000000, rel_diff = 0.000000 <= 0.01
  [+] numeric      | PASS | score=1.000 | severity=info     | adult_test_size = 9769.0
      evidence: lookup_enm(data, adult_test_size) = 9769.000000, claimed = 9769.000000, re

---
## Lab 16.2 --- Generate the Governance Report

Build a markdown-formatted governance report from the pipeline results.  The report includes:

1. **Executive summary** --- dataset, models trained, overall verdict.
2. **Model comparison table** --- accuracy, F1, AUC, train-test gap for each model.
3. **Robustness analysis** --- accuracy under noise for the top 2 models.
4. **Fairness analysis** --- AIR per protected attribute.
5. **Policy compliance summary** --- total policy blocks, tracing status.
6. **Release gate decision** --- approve, conditional or blocked.

In [12]:
report = generate_governance_report(
    phase_results=all_phase_results,
    verification_result=overall_verdict,
    session=session,
    cumulative=cumulative,
)

print(report)

# Governance Report: Adult Income Classification

## Executive Summary

- **Dataset:** Adult Income (OpenML)
- **Models trained:** 5
- **Total tool calls:** 16
- **Knowledge triples extracted:** 156
- **ENM entries stored:** 113
- **Policy blocks:** 0
- **Verification verdict:** PASS (100.0% pass rate)

## Phase Summary

| Phase | Tools | Triples | ENM | Blocks | Time (s) |
|-------|------:|--------:|----:|-------:|---------:|
| DataPhase | 4 | 18 | 7 | 0 | 8.0 |
| TrainingPhase | 6 | 91 | 75 | 0 | 153.3 |
| DiagnosticsPhase | 4 | 30 | 20 | 0 | 16.2 |
| FairnessPhase | 2 | 17 | 11 | 0 | 7.2 |

## Model Comparison

| Model | Type | Accuracy | F1 (macro) | AUC | Train-Test Gap |
|-------|------|----------|------------|-----|----------------|
| LR | LogisticRegression | 0.8185 | 0.6958 | 0.8295 | -0.0046 |
| RF | RandomForestClassifier | 0.8086 | 0.7183 | 0.8272 | 0.1900 |
| GB | GradientBoostingClassifier | 0.8440 | 0.7434 | 0.8676 | -0.0025 |
| DT | DecisionTreeClassifier | 0.8357 | 0.7

### Memory Recall: What Does the Store Know?

Recall entities from the GMS store to show what the pipeline has learned geometrically.  This demonstrates the `recall_entity` API from Chapter 5.

In [13]:
# Recall the top model from memory
top_model_recall = memory.recall_entity(top1)
print(f"=== Memory Recall: {top1} ===")
print(f"  Matched:   {top_model_recall['matched']}")
print(f"  Exists:    {top_model_recall['exists']}")
print(f"  Head triples: {len(top_model_recall['triples_as_head'])}")
print(f"  Tail triples: {len(top_model_recall['triples_as_tail'])}")
print(f"  ENM values:   {len(top_model_recall['enm_values'])}")

# Show some ENM values
if top_model_recall['enm_values']:
    print("\n  Sample ENM values:")
    for key, val in list(top_model_recall['enm_values'].items())[:8]:
        print(f"    {key}: {val}")

# Compare top 2 models geometrically
print(f"\n=== Geometric Comparison: {top2[0]} vs {top2[1]} ===")
comparison = memory.compare_entities(top2[0], top2[1])
print(f"  Tension energy:   {comparison.tension_energy}")
print(f"  Interpretation:   {comparison.interpretation}")
if comparison.metric_deltas:
    print("  Metric deltas:")
    for cat, delta in list(comparison.metric_deltas.items())[:5]:
        print(f"    {cat}: {delta:+.6f}")

=== Memory Recall: GB ===
  Matched:   GB
  Exists:    True
  Head triples: 50
  Tail triples: 1
  ENM values:   27

  Sample ENM values:
    metric::GB_train_accuracy: 0.8415
    metric::GB_train_f1_macro: 0.7426
    metric::GB_train_auc: 0.8716
    metric::GB_test_accuracy: 0.844
    metric::GB_test_f1_macro: 0.7434
    metric::GB_test_auc: 0.8676
    metric::GB_train_test_gap: -0.0025
    ranking::GB_rank: 1.0

=== Geometric Comparison: GB vs DT ===
  Tension energy:   0.9801263809204102
  Interpretation:   irrelevance
  Metric deltas:
    ranking: -1.000000
    robustness: +0.008100
    importance: +0.000676
    metric: +0.001200


### Save the Memory Store

Persist the GMS store to disk so it can be reloaded in later chapters or exercises.

In [14]:
memory.save()
final_stats = memory.memory_stats()
print("Memory store saved.")
print(f"  Path:      {final_stats.get('store_path', STORE_PATH)}")
print(f"  Entities:  {final_stats['entities']}")
print(f"  Relations: {final_stats['relations']}")
print(f"  Triples:   {final_stats['triples']}")
print(f"  ENM:       {final_stats['enm_entries']}")

  Store saved to governance_monograph/lab_stores/ch16_adult_store
Memory store saved.
  Path:      governance_monograph/lab_stores/ch16_adult_store
  Entities:  342
  Relations: 82
  Triples:   526
  ENM:       124


---
## Exercises

### Exercise 16.1 --- Ungoverned Baseline Comparison

Run the same pipeline **without** governance: execute all tool calls directly via `execute_tool` without extraction, memory storage or claim verification.  Compare:

1. Wall-clock time (governed vs ungoverned).
2. How many numeric facts you can recall without ENM.
3. Whether you can detect a fairness violation without the verifier.

**Hint:** Create a plain loop over the same tool sequences but skip `extract_from_tool` and `memory.store_findings`.

### Exercise 16.2 --- Biased Dataset

Modify the Adult Income dataset to amplify gender bias: set `income = 0` for all Female records in the training set.  Re-run the full governed pipeline and observe:

1. Does the fairness phase detect the bias via AIR?
2. What claims fail verification?
3. What release gate decision does the report produce?

**Hint:** After `load_dataset`, modify `session.datasets["adult"]["y_train"]` to zero out female records before running the training phase.

### Exercise 16.3 --- Add a Fifth Phase (Explanation)

Define an `ExplanationPhase` that runs `explain_shap` on the top model.  Wire it into the pipeline after DiagnosticsPhase and before FairnessPhase.  Update the governance report to include a "Feature Attribution" section showing SHAP-based top-5 drivers.

**Hint:** Follow the pattern of `adult_diagnostics_phase()` to create `adult_explanation_phase(top_model)` and add it to the runner.

---
## Lab 16.3 — Audit-Defensible Workflow Hardening

The labs so far showed the four phases compose end-to-end. An auditor's next questions:

1. How do you know the workflow's **governance result reproduces** across runs? → deterministic `WorkflowSnapshot` fingerprint.
2. What happens under **realistic perturbations**? → 6-class adversarial cohort.
3. Do **policy + contract + audit all agree** on every gated event? → cross-component agreement check.
4. Does the bundle **authenticate its producer**? → HMAC-signed `WorkflowBundle` with three refusal paths.

Library: `governed_agent/workflow_hardening/` (8 modules).

### Lab 16.3 §1 — Build a deterministic WorkflowSnapshot

In [ ]:
from knowlytix.harness.governance.workflow_hardening import (
    build_workflow_snapshot, gate_decision_from,
)

# Honest measured pass rate captured from the verifier
release_gate = gate_decision_from(
    overall_status=overall_verdict.overall_status,
    pass_rate=overall_verdict.pass_rate,
    n_blocks=runner.cumulative_stats().get("total_policy_blocks", 0),
)

# Floating metrics (CUDA-trained accuracies) captured separately
# from the fingerprint so byte-identical replay survives CUDA
# non-determinism on individual model accuracies.
floating = {
    f"acc_{name}": float(model.get("test_accuracy", 0.0))
    for name, model in session.models.items()
    if isinstance(model, dict) and "test_accuracy" in model
}

snap = build_workflow_snapshot(
    workflow_id="lab-16-3",
    phase_results=all_phase_results,
    overall_verdict=overall_verdict,
    release_gate=release_gate,
    floating_metrics=floating,
)
print(snap.summary())
print(f"\nfingerprint (full): {snap.fingerprint()}")


### Lab 16.3 §2 — Workflow adversarial cohort (smoke)

In [ ]:
from knowlytix.harness.governance.workflow_hardening import (
    build_workflow_adversarial_cohort_v1,
    sweep_workflow_adversarial,
)

cohort = build_workflow_adversarial_cohort_v1()
print(f"v1 cohort: {len(cohort)} probes")
for probe in cohort:
    print(f"  [{probe.attack_type}] {probe.name}: "
          f"safe={probe.safe_outcomes}")


In [ ]:
# Smoke harness: builds the cohort and confirms each probe has a
# valid mutator + safe-outcomes contract. Full sweep requires
# re-running the workflow per probe (~3 minutes each); deployment
# CI runs the full sweep on a schedule.
report = sweep_workflow_adversarial(cohort)
print(report.summary())


### Lab 16.3 §3 — Build signed WorkflowBundle

In [ ]:
from knowlytix.harness.governance.workflow_hardening import (
    WorkflowOperatingPoint, build_workflow_bundle,
    bind_workflow_from_bundle, measure_phase_latencies,
)

latency = measure_phase_latencies(all_phase_results)
print(latency.summary())
print()

# Operating point ships the *measured* state. A future code fix
# that lifts pass_rate above 0.95 will produce a tighter SLO
# bundle.
op = WorkflowOperatingPoint(
    min_pass_rate=round(snap.pass_rate - 0.01, 6),
    min_cohort_safety_rate=1.0,
    accepted_release_gates=("APPROVED", "CONDITIONAL", "BLOCKED"),
)
bundle = build_workflow_bundle(
    snapshot=snap,
    cohort_report=report,
    per_phase_latency_s=latency.per_phase_dict(),
    attestors_run=("snapshot", "cross_component"),
    operating_point=op,
)
print(f"bundle id: {bundle.bundle_id}")
print(f"signing_method: {bundle.signing_method}")
print(f"signature: {bundle.signature[:24]}...")

# Bind the bundle back to the live snapshot - 3 refusal paths
bind_workflow_from_bundle(bundle, snapshot=snap)
print("\nbound: snapshot fingerprint match + SLO satisfied")


---
## Lab 16.4 — Production Hardening

Five additional gaps closed for the workflow surface:

1. v2 cohort: 6 → 30 probes across 6 attack classes (positions/payloads vary)
2. FSM invariants: 39/39 reachable workflow states pass K1–K5
3. Independent alt-validator: re-derives release-gate verdict via truth table — agreement 100%
4. k-of-n multi-signer workflow (reuses Ch 12 `policy_multisign` API)
5. Latency: snapshot.fingerprint p99 ≈ 20μs, alt_release_gate p99 ≈ 0.1μs

Runbook: `gms_agent_builder_ext/lab16_workflow_runbook.py`. Ship gate: `scripts/gate_workflow_bundle.py`. Model card: `gms_agent_builder_ext/MODEL_CARD_workflow_bundle.md`.

### Lab 16.4 §1 — v2 cohort + FSM invariants

In [ ]:
from knowlytix.harness.governance.workflow_hardening import (
    build_workflow_adversarial_cohort_v2,
    verify_workflow_invariants,
)

cohort_v2 = build_workflow_adversarial_cohort_v2(n_per_type=5)
print(f"v2 cohort: {len(cohort_v2)} probes")
inv = verify_workflow_invariants()
print(inv.summary())


### Lab 16.4 §2 — Alt-validator agreement

In [ ]:
from knowlytix.harness.governance.workflow_hardening import (
    assert_workflow_alt_agreement, alt_release_gate,
)

# Re-derive release-gate verdict from snapshot fingerprint alone
alt = alt_release_gate(
    cumulative_blocks=snap.cumulative_blocks,
    pass_rate=snap.pass_rate,
)
print(f"primary gate: {snap.release_gate}")
print(f"alt gate    : {alt}")
print(f"agree: {alt == snap.release_gate}")

agreement = assert_workflow_alt_agreement([snap])
print(f"\n{agreement.summary()}")


### Lab 16.4 §3 — Multi-signer + latency

In [ ]:
from knowlytix.harness.governance import attach_signatures, sign_for, verify_quorum

keys = {
    "mrm_lead":           "deployment-mrm-lead-key-32-bytes!!",
    "compliance_officer": "deployment-compliance-key-32-bytes!",
    "platform_owner":     "deployment-platform-key-32-bytes!!",
}
sigs = [sign_for(bundle, sid, k) for sid, k in keys.items()]
attach_signatures(bundle, sigs)
quorum = verify_quorum(bundle, keys, k=2)
print(quorum.summary())

print()
from knowlytix.harness.governance.workflow_hardening import (
    bench_snapshot_fingerprint, bench_alt_release_gate,
)
print(bench_snapshot_fingerprint(snap, n_calls=2000, warmup=50).summary())
print(bench_alt_release_gate([snap], n_calls=10000, warmup=100).summary())
